In [26]:
import os
import sys
import numpy as np
import torch
import pandas as pd

# 让notebook能直接import MacroHFT
ROOT = '/home/ustc1958/lxy/129/MacroHFT'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from model.net import subagent

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)

# 两套低层模型根目录（你给的路径）
BASE_TEST = '/home/ustc1958/lxy/129/MacroHFT/result/low_level/ETHUSDT/131testbest_model'
BASE_YUAN = '/home/ustc1958/lxy/129/MacroHFT/result/low_level/ETHUSDT/best_model'

# high_level里n_action=2、hidden=64
N_ACTION = 2
HIDDEN = 64

# 读取特征列表，用来构造网络输入维度
tech_indicator_list = np.load('/home/ustc1958/lxy/129/MacroHFT/data/feature_list/single_features.npy', allow_pickle=True).tolist()
tech_indicator_list_trend = np.load('/home/ustc1958/lxy/129/MacroHFT/data/feature_list/trend_features.npy', allow_pickle=True).tolist()
N_STATE_1 = len(tech_indicator_list)
N_STATE_2 = len(tech_indicator_list_trend)
print('N_STATE_1 =', N_STATE_1, 'N_STATE_2 =', N_STATE_2)

# 构建六个子代理网络（结构必须和训练时一致）
def build_subagents():
    slope_1 = subagent(N_STATE_1, N_STATE_2, N_ACTION, HIDDEN).to(DEVICE)
    slope_2 = subagent(N_STATE_1, N_STATE_2, N_ACTION, HIDDEN).to(DEVICE)
    slope_3 = subagent(N_STATE_1, N_STATE_2, N_ACTION, HIDDEN).to(DEVICE)
    vol_1 = subagent(N_STATE_1, N_STATE_2, N_ACTION, HIDDEN).to(DEVICE)
    vol_2 = subagent(N_STATE_1, N_STATE_2, N_ACTION, HIDDEN).to(DEVICE)
    vol_3 = subagent(N_STATE_1, N_STATE_2, N_ACTION, HIDDEN).to(DEVICE)
    for m in [slope_1, slope_2, slope_3, vol_1, vol_2, vol_3]:
        m.eval()
    return {
        'slope_1': slope_1,
        'slope_2': slope_2,
        'slope_3': slope_3,
        'vol_1': vol_1,
        'vol_2': vol_2,
        'vol_3': vol_3,
    }


def load_weights_v2(agents, base_dir: str):
    """按你当前的目录结构加载：

    {base_dir}/slope/1/best_model.pkl
    {base_dir}/slope/2/best_model.pkl
    {base_dir}/slope/3/best_model.pkl
    {base_dir}/vol/1/best_model.pkl
    {base_dir}/vol/2/best_model.pkl
    {base_dir}/vol/3/best_model.pkl
    """
    mapping = {
        'slope_1': os.path.join(base_dir, 'slope', '1', 'best_model.pkl'),
        'slope_2': os.path.join(base_dir, 'slope', '2', 'best_model.pkl'),
        'slope_3': os.path.join(base_dir, 'slope', '3', 'best_model.pkl'),
        'vol_1': os.path.join(base_dir, 'vol', '1', 'best_model.pkl'),
        'vol_2': os.path.join(base_dir, 'vol', '2', 'best_model.pkl'),
        'vol_3': os.path.join(base_dir, 'vol', '3', 'best_model.pkl'),
    }
    for name, net in agents.items():
        ckpt = mapping[name]
        if not os.path.exists(ckpt):
            raise FileNotFoundError(ckpt)
        net.load_state_dict(torch.load(ckpt, map_location=DEVICE))


# 从数据中抽取一批state/state_trend/previous_action（确保输入分布更真实）
DATA_DIR = '/home/ustc1958/lxy/129/MacroHFT/data/ETHUSDT/whole'
TRAIN_FEATHER = os.path.join(DATA_DIR, 'train.feather')
if not os.path.exists(TRAIN_FEATHER):
    raise FileNotFoundError(TRAIN_FEATHER)

df = pd.read_feather(TRAIN_FEATHER)
print('train rows =', len(df))

missing_1 = [c for c in tech_indicator_list if c not in df.columns]
missing_2 = [c for c in tech_indicator_list_trend if c not in df.columns]
print('missing single features:', len(missing_1))
print('missing trend features:', len(missing_2))
if len(missing_1) or len(missing_2):
    print('示例缺失列(single):', missing_1[:10])
    print('示例缺失列(trend):', missing_2[:10])
    raise RuntimeError('特征列名与feather不匹配，无法自动抽取state；需要你告诉我状态构造逻辑/列名映射。')

B = 512  # batch
idx = np.random.choice(len(df), size=B, replace=False)
state = torch.tensor(df.loc[idx, tech_indicator_list].to_numpy(np.float32), device=DEVICE)
state_trend = torch.tensor(df.loc[idx, tech_indicator_list_trend].to_numpy(np.float32), device=DEVICE)

# subagent.embedding 期望 previous_action 形状为 [B]
previous_action = torch.zeros((B,), dtype=torch.long, device=DEVICE)

print('state', state.shape, 'state_trend', state_trend.shape, 'previous_action', previous_action.shape)

agents_test = build_subagents(); load_weights_v2(agents_test, BASE_TEST)
agents_yuan = build_subagents(); load_weights_v2(agents_yuan, BASE_YUAN)
print('loaded both model sets (test vs yuan)')


def q_stats(agents):
    out = {}
    with torch.no_grad():
        for name, net in agents.items():
            q = net(state, state_trend, previous_action)  # [B, 2]
            out[name] = {
                'mean': float(q.mean().item()),
                'min': float(q.min().item()),
                'max': float(q.max().item()),
                'std': float(q.std().item()),
                'action_gap_mean': float((q[:, 0] - q[:, 1]).abs().mean().item()),
                'action_gap_max': float((q[:, 0] - q[:, 1]).abs().max().item()),
            }
    return out

stats_test = q_stats(agents_test)
stats_yuan = q_stats(agents_yuan)

rows = []
for name in ['slope_1','slope_2','slope_3','vol_1','vol_2','vol_3']:
    t = stats_test[name]
    c = stats_yuan[name]
    rows.append({
        'agent': name,
        'test_mean': t['mean'], 'yuan_mean': c['mean'], 'mean_diff': t['mean'] - c['mean'],
        'test_std': t['std'], 'yuan_std': c['std'], 'std_ratio(test/yuan)': (t['std'] / c['std']) if c['std'] != 0 else np.nan,
        'test_min': t['min'], 'yuan_min': c['min'],
        'test_max': t['max'], 'yuan_max': c['max'],
        'test_gap_mean': t['action_gap_mean'], 'yuan_gap_mean': c['action_gap_mean'],
        'test_gap_max': t['action_gap_max'], 'yuan_gap_max': c['action_gap_max'],
    })

report = pd.DataFrame(rows).sort_values('std_ratio(test/yuan)', ascending=False)
report


device = cuda:0
N_STATE_1 = 36 N_STATE_2 = 9
train rows = 525240
missing single features: 0
missing trend features: 0
state torch.Size([512, 36]) state_trend torch.Size([512, 9]) previous_action torch.Size([512])
loaded both model sets (test vs yuan)


,agent,test_mean,yuan_mean,mean_diff,test_std,yuan_std,std_ratio(test/yuan),test_min,yuan_min,test_max,yuan_max,test_gap_mean,yuan_gap_mean,test_gap_max,yuan_gap_max
2,slope_3,0.657798,0.300905,0.356894,0.308950,0.043580,7.089293,0.322707,-0.033574,7.914497,1.013607,0.068165,0.026039,3.756958,0.561360
0,slope_1,1.663015,2.253573,-0.590558,1.100547,0.172407,6.383420,0.764163,1.952254,29.787487,5.127438,0.306133,0.218436,10.644894,0.421760
3,vol_1,2.111948,3.077799,-0.965852,0.495958,0.204709,2.422751,1.330904,2.680108,5.035486,4.008801,0.337215,0.313129,2.641940,0.741500
1,slope_2,1.672520,2.198317,-0.525797,0.249860,0.133669,1.869241,1.185090,1.891320,5.252149,2.529033,0.301495,0.247983,1.337358,0.530254
5,vol_3,1.413711,1.863261,-0.449551,0.293188,0.167654,1.748771,0.460909,1.547624,7.265129,5.991252,0.250056,0.153425,2.797221,2.382169
4,vol_2,1.470835,1.643243,-0.172408,0.247665,0.262434,0.943724,1.047452,1.209283,4.683887,5.435454,0.237700,0.241021,0.933249,1.419843


In [ ]:
# 只用两套低层(subagent)做验证（test vs yuan）
# 不加载任何高层hyperagent/best_model，避免引入高层权重干扰。
# 目标：用一致的组合规则对比两套低层输出的“组合Q分布/softmax尖锐度”，并做 slope_3 消融定位。

import torch
import numpy as np
import pandas as pd

names = ['slope_1','slope_2','slope_3','vol_1','vol_2','vol_3']

@torch.no_grad()
def stack_q(agents):
    qs = []
    for n in names:
        q = agents[n](state, state_trend, previous_action)  # [B,2]
        qs.append(q.unsqueeze(1))
    return torch.cat(qs, dim=1)  # [B,6,2]

@torch.no_grad()
def summarize(q_b2, tag=''):
    # 尖锐度：softmax最大概率的均值（越接近1越尖）
    p = torch.softmax(q_b2, dim=1)
    pmax = p.max(dim=1).values
    gap = (q_b2[:, 0] - q_b2[:, 1]).abs()
    return {
        'tag': tag,
        'q_mean': float(q_b2.mean().item()),
        'q_std': float(q_b2.std().item()),
        'q_min': float(q_b2.min().item()),
        'q_max': float(q_b2.max().item()),
        'gap_mean': float(gap.mean().item()),
        'gap_max': float(gap.max().item()),
        'softmax_pmax_mean': float(pmax.mean().item()),
        'softmax_pmax_p95': float(torch.quantile(pmax, 0.95).item()),
    }

@torch.no_grad()
def combine_with_w(qs_b62, w_b6):
    return (qs_b62 * w_b6.unsqueeze(-1)).sum(dim=1)  # [B,2]

@torch.no_grad()
def w_uniform(B):
    return torch.full((B, 6), 1.0 / 6.0, device=DEVICE, dtype=torch.float32)

@torch.no_grad()
def w_by_gap_softmax(qs_b62, tau=1.0):
    # 用每个子代理自己的 action gap 作为“置信度”，对6个代理做softmax归一化
    # gap越大 => 权重越大。tau越小越尖。
    gaps = (qs_b62[:, :, 0] - qs_b62[:, :, 1]).abs()  # [B,6]
    logits = gaps / float(tau)
    return torch.softmax(logits, dim=1)  # [B,6]

@torch.no_grad()
def w_group_uniform(kind='all'):
    # kind='slope' => 仅 slope_1..3 均匀（vol权重为0）
    # kind='vol'   => 仅 vol_1..3 均匀（slope权重为0）
    w = torch.zeros((B, 6), device=DEVICE, dtype=torch.float32)
    if kind == 'slope':
        w[:, 0:3] = 1.0 / 3.0
    elif kind == 'vol':
        w[:, 3:6] = 1.0 / 3.0
    else:
        w[:, :] = 1.0 / 6.0
    return w

@torch.no_grad()
def top1_hist(w_b6, tag=''):
    top1 = w_b6.argmax(dim=1)  # [B]
    counts = torch.bincount(top1, minlength=6).float()
    frac = counts / counts.sum()
    return pd.DataFrame({
        'agent': names,
        f'{tag}_top1_frac': [float(frac[i].item()) for i in range(6)],
    }).sort_values(f'{tag}_top1_frac', ascending=False)

@torch.no_grad()
def extreme_case_report(qs_b62, w_b6, tag='', k=10):
    # 找“最极端的样本”，看是谁在支配（按 w 的top1，以及对应gap/qmax贡献）
    gaps = (qs_b62[:, :, 0] - qs_b62[:, :, 1]).abs()      # [B,6]
    qmax = qs_b62.max(dim=2).values                        # [B,6]
    top1 = w_b6.argmax(dim=1)                              # [B]

    # 两类极端：最大 gap / 最大 qmax （按组合前单代理）
    idx_gap = gaps.max(dim=1).values.topk(k=min(k, gaps.size(0))).indices
    idx_qmx = qmax.max(dim=1).values.topk(k=min(k, qmax.size(0))).indices

    def _one(idxs, kind):
        sub = []
        for i in idxs.tolist():
            j = int(top1[i].item())
            sub.append({
                'kind': kind,
                'sample_i': i,
                'top1_agent': names[j],
                'top1_w': float(w_b6[i, j].item()),
                'agent_gap_max': float(gaps[i].max().item()),
                'agent_qmax_max': float(qmax[i].max().item()),
                'top1_agent_gap': float(gaps[i, j].item()),
                'top1_agent_qmax': float(qmax[i, j].item()),
            })
        return sub

    df = pd.DataFrame(_one(idx_gap, 'max_gap') + _one(idx_qmx, 'max_qmax'))
    return df

# --- 1) 计算两套低层的6个Q ---
qs_test = stack_q(agents_test)
qs_yuan = stack_q(agents_yuan)

print('qs_test shape:', tuple(qs_test.shape), 'qs_yuan shape:', tuple(qs_yuan.shape))

# --- 2) 方案A：均匀权重组合（最干净的对比） ---
w_u = w_uniform(B)
comb_test_u = combine_with_w(qs_test, w_u)
comb_yuan_u = combine_with_w(qs_yuan, w_u)
print('\n== Uniform-weight ensemble ==')
print(summarize(comb_test_u, 'test low + uniform w'))
print(summarize(comb_yuan_u, 'yuan low + uniform w'))

# --- 3) 方案B：按|gap|自适应权重（看“谁更极端就更被放大”） ---
# 注：这是一个与高层无关的“代理选择器”替代，用来模拟 mixture-of-experts 的放大效应。
for tau in [2.0, 1.0, 0.5, 0.25]:
    w_test_gap = w_by_gap_softmax(qs_test, tau=tau)
    w_yuan_gap = w_by_gap_softmax(qs_yuan, tau=tau)
    comb_test_gap = combine_with_w(qs_test, w_test_gap)
    comb_yuan_gap = combine_with_w(qs_yuan, w_yuan_gap)
    print(f"\n== Gap-softmax ensemble (tau={tau}) ==")
    print(summarize(comb_test_gap, f'test low + gap-softmax(tau={tau})'))
    print(summarize(comb_yuan_gap, f'yuan low + gap-softmax(tau={tau})'))

# 同时报告 gap-softmax 下平均权重（看是否 slope_3 被强烈偏好）
print('\n== Mean weights under gap-softmax (tau=1.0) ==')
w_test_gap1 = w_by_gap_softmax(qs_test, tau=1.0)
w_yuan_gap1 = w_by_gap_softmax(qs_yuan, tau=1.0)
df_w = pd.DataFrame({
    'agent': names,
    'w_test_mean': [float(w_test_gap1[:, i].mean().item()) for i in range(6)],
    'w_yuan_mean': [float(w_yuan_gap1[:, i].mean().item()) for i in range(6)],
    'ratio(test/yuan)': [float((w_test_gap1[:, i].mean() / (w_yuan_gap1[:, i].mean() + 1e-12)).item()) for i in range(6)],
}).sort_values('ratio(test/yuan)', ascending=False)
display(df_w)

# --- 3.1) 样本级别归因：谁最常成为top1权重？极端样本由谁支配？ ---
print('\n== Top1 agent frequency under gap-softmax (tau=1.0) ==')
display(top1_hist(w_test_gap1, tag='test'))
display(top1_hist(w_yuan_gap1, tag='yuan'))

print('\n== Extreme samples attribution under gap-softmax (tau=1.0) ==')
# k 设小一点，避免输出过大
ext_test = extreme_case_report(qs_test, w_test_gap1, tag='test', k=8)
ext_yuan = extreme_case_report(qs_yuan, w_yuan_gap1, tag='yuan', k=8)
print('test extreme cases (first rows):')
display(ext_test.head(16))
print('yuan extreme cases (first rows):')
display(ext_yuan.head(16))

# --- 4) 方案C：只看 slope 组 / vol 组的组合输出（定位是哪一组导致尖锐） ---
for kind in ['slope', 'vol']:
    w_g = w_group_uniform(kind)
    comb_test_g = combine_with_w(qs_test, w_g)
    comb_yuan_g = combine_with_w(qs_yuan, w_g)
    print(f"\n== Group-only uniform ensemble ({kind}) ==")
    print(summarize(comb_test_g, f'test low + {kind}-only'))
    print(summarize(comb_yuan_g, f'yuan low + {kind}-only'))

# --- 5) 消融：在 test 低层下仅替换 slope_3 为 yuan 版本 ---
qs_test_replace_s3 = qs_test.clone()
qs_test_replace_s3[:, 2, :] = qs_yuan[:, 2, :]

comb_test_u_r = combine_with_w(qs_test_replace_s3, w_u)
print('\n== Ablation (uniform w): test low but slope_3 <- yuan ==')
print(summarize(comb_test_u_r, 'test low(s3<-yuan) + uniform w'))

w_test_gap1_r = w_by_gap_softmax(qs_test_replace_s3, tau=1.0)
comb_test_gap1_r = combine_with_w(qs_test_replace_s3, w_test_gap1_r)
print('\n== Ablation (gap-softmax tau=1.0): test low but slope_3 <- yuan ==')
print(summarize(comb_test_gap1_r, 'test low(s3<-yuan) + gap-softmax(tau=1.0)'))

# slope_3 的单项输出统计（对齐你之前的怀疑）
contrib_s3_test_u = qs_test[:, 2, :] * (1.0/6.0)
contrib_s3_yuan_u = qs_yuan[:, 2, :] * (1.0/6.0)
print('\n== slope_3 contribution under uniform w (w=1/6) ==')
print('test slope_3 contrib:', summarize(contrib_s3_test_u, 'contrib_s3(test,uniform)'))
print('yuan slope_3 contrib:', summarize(contrib_s3_yuan_u, 'contrib_s3(yuan,uniform)'))


qs_test shape: (512, 6, 2) qs_yuan shape: (512, 6, 2)

== Uniform-weight ensemble ==
{'tag': 'test low + uniform w', 'q_mean': 1.4983044862747192, 'q_std': 0.38541820645332336, 'q_min': 1.1333937644958496, 'q_max': 8.215622901916504, 'gap_mean': 0.23683345317840576, 'gap_max': 0.6471493244171143, 'softmax_pmax_mean': 0.5588474273681641, 'softmax_pmax_p95': 0.5933845639228821}
{'tag': 'yuan low + uniform w', 'q_mean': 1.8895163536071777, 'q_std': 0.13655097782611847, 'q_min': 1.6891257762908936, 'q_max': 3.7148900032043457, 'gap_mean': 0.19584164023399353, 'gap_max': 0.5409278869628906, 'softmax_pmax_mean': 0.5487606525421143, 'softmax_pmax_p95': 0.5772857069969177}

== Gap-softmax ensemble (tau=2.0) ==
{'tag': 'test low + gap-softmax(tau=2.0)', 'q_mean': 1.5466746091842651, 'q_std': 1.0038833618164062, 'q_min': 1.1447066068649292, 'q_max': 27.70057487487793, 'gap_mean': 0.2619302272796631, 'gap_max': 9.566173553466797, 'softmax_pmax_mean': 0.5613881349563599, 'softmax_pmax_p95': 0.5970

,agent,w_test_mean,w_yuan_mean,ratio(test/yuan)
5,vol_3,0.165977,0.158586,1.046606
0,slope_1,0.175039,0.168994,1.035766
1,slope_2,0.175277,0.174047,1.007066
2,slope_3,0.138103,0.139443,0.990385
3,vol_1,0.181214,0.186053,0.973992
4,vol_2,0.164391,0.172877,0.950915



== Top1 agent frequency under gap-softmax (tau=1.0) ==


,agent,test_top1_frac
3,vol_1,0.533203
0,slope_1,0.214844
1,slope_2,0.160156
5,vol_3,0.064453
4,vol_2,0.025391
2,slope_3,0.001953


,agent,yuan_top1_frac
3,vol_1,0.703125
1,slope_2,0.193359
4,vol_2,0.074219
0,slope_1,0.027344
5,vol_3,0.001953
2,slope_3,0.000000



== Extreme samples attribution under gap-softmax (tau=1.0) ==
test extreme cases (first rows):


,kind,sample_i,top1_agent,top1_w,agent_gap_max,agent_qmax_max,top1_agent_gap,top1_agent_qmax
0,max_gap,156,slope_1,0.998107,10.644894,29.787487,10.644894,29.787487
1,max_gap,405,slope_1,0.503670,1.817974,7.290921,1.817974,7.290921
2,max_gap,142,vol_3,0.279847,1.228680,2.963738,1.228680,1.689589
3,max_gap,72,slope_1,0.324495,1.086851,2.088661,1.086851,1.851014
4,max_gap,215,vol_3,0.271752,0.944456,3.061394,0.944456,2.772618
5,max_gap,31,vol_1,0.281340,0.909705,4.537928,0.909705,4.537928
6,max_gap,346,vol_2,0.220836,0.842698,4.112479,0.842698,2.315174
7,max_gap,482,vol_3,0.217531,0.774823,3.864173,0.774823,2.855612
8,max_qmax,156,slope_1,0.998107,10.644894,29.787487,10.644894,29.787487
9,max_qmax,405,slope_1,0.503670,1.817974,7.290921,1.817974,7.290921


yuan extreme cases (first rows):


,kind,sample_i,top1_agent,top1_w,agent_gap_max,agent_qmax_max,top1_agent_gap,top1_agent_qmax
0,max_gap,156,vol_3,0.539225,2.382169,5.991252,2.382169,5.991252
1,max_gap,457,vol_2,0.303315,0.970631,3.585126,0.970631,3.559021
2,max_gap,346,vol_2,0.234416,0.757595,3.812819,0.757595,2.504167
3,max_gap,405,vol_1,0.227413,0.741500,3.580230,0.741500,3.580230
4,max_gap,482,vol_1,0.249949,0.718368,3.407476,0.718368,3.407476
5,max_gap,128,vol_1,0.224514,0.679854,3.755774,0.679854,3.755774
6,max_gap,422,vol_1,0.225085,0.656933,3.776504,0.656933,3.776504
7,max_gap,142,vol_2,0.207317,0.641506,3.361724,0.641506,2.183764
8,max_qmax,156,vol_3,0.539225,2.382169,5.991252,2.382169,5.991252
9,max_qmax,269,vol_1,0.211949,0.521825,4.008801,0.521825,4.008801



== Group-only uniform ensemble (slope) ==
{'tag': 'test low + slope-only', 'q_mean': 1.3311110734939575, 'q_std': 0.5142802596092224, 'q_min': 0.939856231212616, 'q_max': 12.619939804077148, 'gap_mean': 0.2115878015756607, 'gap_max': 1.8501930236816406, 'softmax_pmax_mean': 0.5524265766143799, 'softmax_pmax_p95': 0.5878172516822815}
{'tag': 'yuan low + slope-only', 'q_mean': 1.5842647552490234, 'q_std': 0.09483982622623444, 'q_min': 1.3904162645339966, 'q_max': 2.590956449508667, 'gap_mean': 0.15805105865001678, 'gap_max': 0.34419727325439453, 'softmax_pmax_mean': 0.5394114255905151, 'softmax_pmax_p95': 0.5607085227966309}

== Group-only uniform ensemble (vol) ==
{'tag': 'test low + vol-only', 'q_mean': 1.665497899055481, 'q_std': 0.31177327036857605, 'q_min': 1.24483323097229, 'q_max': 5.661500453948975, 'gap_mean': 0.2707405090332031, 'gap_max': 2.124136447906494, 'softmax_pmax_mean': 0.5669074654579163, 'softmax_pmax_p95': 0.6035458445549011}
{'tag': 'yuan low + vol-only', 'q_mean'

: 